# 量化交易入门 Vol.5：qlib 完整工作流

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/greathousesh/qlora-sft-tutorial/blob/main/quant/05_qlib_full_pipeline.ipynb)

> **Kaggle 一键运行**：点击上方按钮 → 选择 **"CPU"** → Run All

## 本节你将学到

| 知识点 | 说明 |
|--------|------|
| **qlib 架构** | 数据层、特征层、模型层、回测层的设计 |
| **qlib 数据初始化** | 如何用 qlib 管理金融数据 |
| **DataHandler** | qlib 的特征工程管道 |
| **DatasetH** | 带时间划分的数据集 |
| **qlib LightGBM** | 用 qlib 的内置模型训练 |
| **qlib 回测** | Strategy + Executor + Collector |
| **Recorder** | 实验管理与结果复现 |

## 为什么需要 qlib？

前 4 节我们用 pandas 手写了整个流程。好处是理解原理，但代价是：
- 因子定义混乱（没有统一语法）
- 数据管理麻烦（多次重复下载、处理）
- 回测代码复杂（容易出 bug）
- 实验不可复现（超参数散落各处）

**qlib 的价值**：微软开源的量化研究框架，系统性解决以上问题。

```
┌─────────────────────────────────────────────────────┐
│                    qlib 架构                          │
│                                                       │
│  数据层          DataProvider                         │
│    │             - 本地存储（高效二进制格式）            │
│    │             - 表达式引擎（自动向量化计算）           │
│    ↓                                                  │
│  特征层          DataHandler                          │
│    │             - 统一的特征定义与处理                  │
│    │             - 自动 normalization/fillna             │
│    ↓                                                  │
│  数据集          DatasetH                             │
│    │             - 训练/验证/测试时间切分                │
│    ↓                                                  │
│  模型层          Model (LightGBM, LSTM, etc.)         │
│    │             - 统一接口，可互换模型                  │
│    ↓                                                  │
│  回测层          Backtest Engine                      │
│                  - 组合优化 + 模拟执行                  │
└─────────────────────────────────────────────────────┘
```

## Step 0：安装 qlib

In [ ]:
import subprocess, sys

# qlib 依赖 tables（HDF5）和 lightgbm
pkgs = [
    "pyqlib",
    "tables>=3.6.1",       # HDF5 存储后端
    "lightgbm>=3.3.0",
    "ruamel.yaml>=0.16",   # qlib 配置文件解析
    "fire>=0.4.0",          # qlib CLI 工具
    "pandas>=1.5.0",
    "numpy>=1.23.0",
    "matplotlib>=3.6.0",
    "scipy>=1.9.0",
]

print("安装 qlib 及依赖（约 1-2 分钟）...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=True)
print("✅ 安装完毕")

## Step 1：准备 qlib 数据

qlib 使用自己的二进制格式存储数据（比 CSV 快 10-100 倍）。

有两种方式获取数据：
1. **下载 qlib 预打包数据**（推荐）：官方提供 CSI300 / US市场 数据集
2. **自己转换**：用 yfinance 下载数据，再用 qlib 的脚本转换格式

我们采用方式 2（Kaggle 上网络受限时更稳定），并展示两种方式的代码。

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams.update({'figure.dpi': 100, 'font.size': 11,
                     'axes.titlesize': 12, 'axes.grid': True, 'grid.alpha': 0.3})

# ── 方式一：下载 qlib 官方预打包数据（如果网络允许）─────────────────────
# from qlib.tests.data import GetData
# GetData().qlib_data(
#     target_dir='~/.qlib/qlib_data/us_data',
#     region='us',
#     interval='1d',
#     exists_skip=True
# )
# print("官方数据下载完毕")

# ── 方式二：用 yfinance 下载数据并转换为 qlib 格式 ────────────────────
# 这是我们在 Kaggle 上使用的方式

TICKERS = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN',
           'TSLA', 'JPM', 'GS', 'JNJ', 'WMT']
START, END = '2018-01-01', '2024-01-01'
DATA_DIR   = os.path.expanduser('~/.qlib/qlib_data/custom_us')

print(f"下载股票数据 ({START} ~ {END})...")
raw = yf.download(TICKERS, start=START, end=END, auto_adjust=True, progress=False)

prices  = raw['Close'].dropna(how='all')[TICKERS]
volumes = raw['Volume'].dropna(how='all')[TICKERS]
highs   = raw['High'].dropna(how='all')[TICKERS]
lows    = raw['Low'].dropna(how='all')[TICKERS]
opens   = raw['Open'].dropna(how='all')[TICKERS]

print(f"数据下载完毕: {prices.shape[0]} 交易日 × {len(TICKERS)} 只股票")

# 转换成 qlib 格式：每只股票一个目录，包含 features 文件
def create_qlib_data(prices, highs, lows, opens, volumes, output_dir):
    """
    将 OHLCV 数据写入 qlib 所需的目录结构：
    output_dir/
        calendars/
            day.txt         <- 所有交易日列表
        instruments/
            all.txt         <- 所有股票代码
        features/
            AAPL/
                close.day.bin   <- 二进制格式（qlib 会自动处理）
            ...
    """
    import struct
    
    os.makedirs(f'{output_dir}/calendars',   exist_ok=True)
    os.makedirs(f'{output_dir}/instruments', exist_ok=True)
    
    tickers = list(prices.columns)
    all_dates = prices.index.strftime('%Y-%m-%d').tolist()
    
    # calendars/day.txt
    with open(f'{output_dir}/calendars/day.txt', 'w') as f:
        f.write('\n'.join(all_dates))
    
    # instruments/all.txt
    with open(f'{output_dir}/instruments/all.txt', 'w') as f:
        for t in tickers:
            f.write(f'{t}\t{all_dates[0]}\t{all_dates[-1]}\n')
    
    # features/
    for ticker in tickers:
        feat_dir = f'{output_dir}/features/{ticker.lower()}'
        os.makedirs(feat_dir, exist_ok=True)
        
        fields = {
            'close':  prices[ticker],
            'high':   highs[ticker],
            'low':    lows[ticker],
            'open':   opens[ticker],
            'volume': volumes[ticker],
        }
        
        for fname, series in fields.items():
            vals = series.reindex(prices.index).values.astype(np.float32)
            # qlib binary format: 4-byte float array
            with open(f'{feat_dir}/{fname}.day.bin', 'wb') as f:
                f.write(struct.pack(f'{len(vals)}f', *vals))
    
    print(f"  qlib 数据目录创建完成: {output_dir}")
    print(f"  股票数: {len(tickers)}, 交易日数: {len(all_dates)}")

create_qlib_data(prices, highs, lows, opens, volumes, DATA_DIR)

## Step 2：初始化 qlib

qlib 使用全局初始化，之后所有查询都会从指定目录读取数据。

In [ ]:
import qlib
from qlib.constant import REG_US

print("初始化 qlib...")
qlib.init(
    provider_uri=DATA_DIR,
    region=REG_US,
)
print("✅ qlib 初始化完毕")
print(f"  数据目录: {DATA_DIR}")
print(f"  区域设置: {REG_US}")

## Step 3：使用 qlib 数据层

`qlib.data.D` 是 qlib 的核心数据接口。  
它支持强大的**表达式引擎**，可以用简洁的公式语言定义因子。

In [ ]:
from qlib.data import D

# ── 3a：基础数据查询 ────────────────────────────────────────────────────
print("[3a] 查询原始 OHLCV 数据：")
df_basic = D.features(
    instruments=TICKERS[:3],
    fields=['$close', '$volume', '$high', '$low', '$open'],
    start_time='2022-01-01',
    end_time='2022-12-31',
    freq='day'
)
print(df_basic.head(8).to_string())
print(f"\n形状: {df_basic.shape}")

In [ ]:
# ── 3b：qlib 表达式引擎——用公式定义因子 ─────────────────────────────────
# 这就是 qlib 最强大的地方：用类 Excel 公式语言自动计算因子

print("[3b] qlib 表达式引擎因子计算：")

factor_exprs = {
    # ── 基础 ──
    '日收益率':       '$close / Ref($close, 1) - 1',

    # ── 动量族 ──
    '动量_1月':       '$close / Ref($close, 21) - 1',
    '动量_3月':       '$close / Ref($close, 63) - 1',
    '动量_12m_skip1': 'Ref($close, 21) / Ref($close, 252) - 1',

    # ── 波动族 ──
    '波动_20d':       'Std(Log($close / Ref($close, 1)), 20) * 16',  # 年化：×√252≈16
    '波动_60d':       'Std(Log($close / Ref($close, 1)), 60) * 16',

    # ── 均线族 ──
    '均线20d':        'Mean($close, 20)',
    '价格位置':       '$close / Mean($close, 60) - 1',  # 当前价格相对60日均线偏离

    # ── 成交量族 ──
    '量比_5d60d':     'Mean($volume, 5) / Mean($volume, 60)',
    '量价相关_20d':   'Corr($close, $volume, 20)',

    # ── 技术指标 ──
    '日内振幅':       '($high - $low) / $close',
    '上影线':         '($high - Greater($close, $open)) / ($close + 1e-6)',
    '下影线':         '(Lesser($close, $open) - $low)  / ($close + 1e-6)',

    # ── 高阶特征 ──
    '价格加速度':     '$close / Ref($close, 5) - Ref($close, 5) / Ref($close, 10)',
    '收益偏度_20d':   'Skew(Log($close / Ref($close, 1)), 20)',
}

df_factors = D.features(
    instruments=TICKERS,
    fields=list(factor_exprs.values()),
    start_time='2019-01-01',
    end_time='2024-01-01',
    freq='day'
)
df_factors.columns = list(factor_exprs.keys())

print(f"因子矩阵形状: {df_factors.shape}")
print(f"股票 × 日期 × 因子数: {len(TICKERS)} × ~{df_factors.shape[0]//len(TICKERS)} × {df_factors.shape[1]}")
print("\n前几行：")
print(df_factors.head(5).round(4).to_string())

In [ ]:
# 可视化：qlib 因子数据探索
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

# 选几个有代表性的因子展示
show_factors = ['日收益率', '动量_3月', '波动_60d', '量比_5d60d', '量价相关_20d', '价格加速度']

for ax, fname in zip(axes, show_factors):
    # 每只股票的因子时序
    factor_series = df_factors[fname].unstack(level='instrument') if 'instrument' in df_factors.index.names else None
    
    # 如果 index 是 MultiIndex (instrument, datetime)
    try:
        for ticker in TICKERS[:4]:
            try:
                ts = df_factors.xs(ticker, level='instrument')[fname].dropna()
                if hasattr(ts.index, 'get_level_values'):
                    ts.index = ts.index.get_level_values(-1)
                ax.plot(ts.index, ts, lw=1, alpha=0.7, label=ticker)
            except Exception:
                pass
    except Exception:
        pass
    
    ax.set_title(fname)
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.suptitle('qlib 表达式引擎计算的因子时序', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('qlib_factors.png', bbox_inches='tight')
plt.show()
print("因子时序可视化完成")

## Step 4：DataHandler —— qlib 的特征工程管道

`DataHandler` 是 qlib 中专门处理特征工程的组件。它负责：
1. **特征定义**：用表达式引擎定义所有特征
2. **标签定义**：定义预测目标（未来收益）
3. **数据清洗**：处理缺失值、异常值
4. **归一化**：截面标准化、时序标准化

DataHandler 把数据处理逻辑和模型训练逻辑解耦，让代码更清晰。

In [ ]:
from qlib.contrib.data.handler import Alpha158
from qlib.data.dataset.handler import DataHandlerLP
from qlib.data.dataset import DatasetH

# ── 方式一：使用 qlib 内置的 Alpha158 因子库 ─────────────────────────
# Alpha158 包含微软研究院精心设计的 158 个技术因子
# 这些因子在学术论文和实际交易中都被验证有效

print("[4a] 初始化 Alpha158 DataHandler...")
try:
    handler_config = {
        'start_time': '2019-01-01',
        'end_time':   '2023-12-31',
        'fit_start_time': '2019-01-01',
        'fit_end_time':   '2021-12-31',
        'instruments': TICKERS,
    }
    
    handler = Alpha158(**handler_config)
    df_alpha158 = handler.fetch(col_set='feature')
    print(f"Alpha158 特征矩阵: {df_alpha158.shape}")
    print(f"特征数量: {df_alpha158.shape[1]}")
    print(f"特征名示例: {list(df_alpha158.columns[:10])}")
except Exception as e:
    print(f"Alpha158 加载提示: {e}")
    print("(Alpha158 需要完整数据，我们将使用自定义 DataHandler 代替)")

print("\n[4b] 使用自定义 DataHandler：")

In [ ]:
# ── 自定义 DataHandler（适合我们的自定义数据）────────────────────────────
# 定义特征和标签

FEATURE_CONFIG = [
    # 动量类
    ("$close / Ref($close, 21) - 1",   "mom_1m"),
    ("$close / Ref($close, 63) - 1",   "mom_3m"),
    ("Ref($close,21) / Ref($close,252) - 1", "mom_12m_skip1m"),
    
    # 反转类
    ("1 - $close / Ref($close, 5)",    "rev_1w"),
    ("1 - $close / Ref($close, 21)",   "rev_1m"),
    
    # 波动类
    ("Std(Log($close/Ref($close,1)), 20) * 16", "vol_20d"),
    ("Std(Log($close/Ref($close,1)), 60) * 16", "vol_60d"),
    ("($high - $low) / $close",         "intraday_range"),
    
    # 成交量类
    ("Mean($volume, 5) / Mean($volume, 60)",  "vol_ratio_5_60"),
    ("Corr($close, $volume, 20)",              "price_vol_corr"),
    
    # 价格位置
    ("$close / Mean($close, 20) - 1",  "price_vs_ma20"),
    ("$close / Mean($close, 60) - 1",  "price_vs_ma60"),
    ("($close - Min($close, 60)) / (Max($close, 60) - Min($close, 60) + 1e-6)", "price_pos_60d"),
]

LABEL_CONFIG = [
    # 未来 10 天收益（标签）
    ("Ref($close, -10) / $close - 1",  "label_10d"),
]

print("自定义特征清单：")
for i, (expr, name) in enumerate(FEATURE_CONFIG, 1):
    print(f"  [{i:2d}] {name:<20s}: {expr}")

print(f"\n标签：{LABEL_CONFIG[0][1]} = 未来10天收益率")

In [ ]:
# 使用 qlib 的低级 API 直接查询带归一化的特征
from qlib.data.dataset.processor import RobustZScoreNorm, Fillna

# 查询原始特征
feature_exprs = [expr for expr, _ in FEATURE_CONFIG]
feature_names = [name for _, name in FEATURE_CONFIG]
label_exprs   = [expr for expr, _ in LABEL_CONFIG]
label_names   = [name for _, name in LABEL_CONFIG]

print("查询特征数据...")
df_feat = D.features(
    instruments=TICKERS,
    fields=feature_exprs + label_exprs,
    start_time='2019-01-01',
    end_time='2023-12-31',
    freq='day'
)
df_feat.columns = feature_names + label_names

print(f"特征矩阵: {df_feat.shape}")
print(f"  行数: {df_feat.shape[0]} (股票×日期)")
print(f"  列数: {df_feat.shape[1]} (特征+标签)")
print(f"  缺失率: {df_feat.isna().mean().mean()*100:.1f}%")

# 展示数据结构
print("\n多级索引结构：")
print(df_feat.index[:4])
print("\n数据预览：")
print(df_feat.head(3).round(4).to_string())

## Step 5：qlib LightGBM 模型训练

利用 qlib 的模型接口训练 LightGBM，实现与 Vol.3 相同的功能，但代码更简洁。

In [ ]:
from scipy.stats import spearmanr
from sklearn.preprocessing import RobustScaler
import lightgbm as lgb

# 整理 panel 数据
# qlib 的 MultiIndex: level 0 = instrument, level 1 = datetime (或反之)
if 'instrument' in df_feat.index.names:
    df_feat_reset = df_feat.reset_index()
else:
    df_feat_reset = df_feat.copy()
    df_feat_reset = df_feat_reset.reset_index()

# 重命名列
if 'instrument' in df_feat_reset.columns:
    df_feat_reset = df_feat_reset.rename(columns={'instrument': 'ticker', 'datetime': 'date'})
elif 'level_0' in df_feat_reset.columns and 'level_1' in df_feat_reset.columns:
    df_feat_reset.columns.values[0] = 'ticker'
    df_feat_reset.columns.values[1] = 'date'

df_feat_reset = df_feat_reset.dropna(subset=feature_names + label_names)
df_feat_reset['date'] = pd.to_datetime(df_feat_reset['date'])

print(f"Panel 数据: {len(df_feat_reset)} 行")
print(f"日期范围: {df_feat_reset['date'].min().date()} ~ {df_feat_reset['date'].max().date()}")
print(f"股票数: {df_feat_reset['ticker'].nunique()}")

In [ ]:
# Walk-Forward 训练（与 Vol.3 相同，但使用 qlib 的特征）
TRAIN_END = pd.Timestamp('2021-12-31')
TEST_END  = pd.Timestamp('2023-12-31')

train_df = df_feat_reset[df_feat_reset['date'] <= TRAIN_END]
test_df  = df_feat_reset[(df_feat_reset['date'] > TRAIN_END) & 
                          (df_feat_reset['date'] <= TEST_END)]

X_train = train_df[feature_names]
y_train = train_df['label_10d']
X_test  = test_df[feature_names]
y_test  = test_df['label_10d']

print(f"训练集: {len(X_train)} 行, 测试集: {len(X_test)} 行")

# 截面 rank 标准化标签（更稳定的训练目标）
train_df = train_df.copy()
train_df['label_rank'] = train_df.groupby('date')['label_10d'].rank(pct=True) - 0.5
y_train_rank = train_df['label_rank']

test_df = test_df.copy()
test_df['label_rank']  = test_df.groupby('date')['label_10d'].rank(pct=True) - 0.5
y_test_rank = test_df['label_rank']

# 训练 LightGBM
lgb_params = {
    'objective': 'regression',
    'metric': 'mse',
    'num_leaves': 64,
    'learning_rate': 0.02,
    'n_estimators': 300,
    'subsample': 0.7,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.2,
    'min_child_samples': 20,
    'verbose': -1,
    'random_state': 42,
}

print("\n训练 LightGBM 模型...")
model = lgb.LGBMRegressor(**lgb_params)
model.fit(
    X_train, y_train_rank,
    eval_set=[(X_test, y_test_rank)],
    callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(period=50)]
)
print("训练完成！")

# 预测
test_df = test_df.copy()
test_df['pred'] = model.predict(X_test)

# 计算 Rank IC
daily_ic = test_df.groupby('date').apply(
    lambda g: spearmanr(g['pred'], g['label_10d'])[0] if len(g) >= 3 else np.nan
).dropna()

print(f"\n模型评估（测试集 {TRAIN_END.date()} → {TEST_END.date()}）：")
print(f"  Rank IC 均值: {daily_ic.mean():.4f}")
print(f"  ICIR:         {daily_ic.mean()/daily_ic.std():.3f}")
print(f"  IC>0 占比:    {(daily_ic>0).mean()*100:.1f}%")

## Step 6：qlib 回测框架

qlib 的回测框架与我们在 Vol.4 手写的相比，主要优势：
1. **更真实的交易模拟**：考虑 A 股涨跌停板、T+1 交收等细节
2. **组合优化**：内置均值方差优化、风险因子中性化
3. **报告自动生成**：标准化绩效报告

下面演示 qlib 回测 API 的使用方式。

In [ ]:
# 展示 qlib 回测框架的标准用法（代码模板，可在有完整 qlib 数据时直接运行）
qlib_backtest_template = '''
# ══════════════════════════════════════════════════════════════
# qlib 完整回测流程（使用官方数据时直接可运行）
# ══════════════════════════════════════════════════════════════

import qlib
from qlib.constant import REG_US
from qlib.contrib.model.gbdt import LGBModel
from qlib.contrib.data.handler import Alpha158
from qlib.data.dataset import DatasetH
from qlib.contrib.strategy import TopkDropoutStrategy
from qlib.backtest import backtest, executor as exec
from qlib.contrib.evaluate import risk_analysis
from qlib.contrib.report import analysis_model, analysis_position

# 1. 初始化
qlib.init(provider_uri="~/.qlib/qlib_data/us_data", region=REG_US)

# 2. 数据集配置
dataset = DatasetH(
    handler={
        "class": "Alpha158",
        "kwargs": {
            "start_time": "2018-01-01",
            "end_time":   "2023-12-31",
            "fit_start_time": "2018-01-01",
            "fit_end_time":   "2020-12-31",
            "instruments": "sp500",  # 或自定义列表
        }
    },
    segments={
        "train": ("2018-01-01", "2020-12-31"),
        "valid": ("2021-01-01", "2021-12-31"),
        "test":  ("2022-01-01", "2023-12-31"),
    }
)

# 3. 模型训练
model = LGBModel({
    "num_leaves": 64,
    "learning_rate": 0.02,
    "n_estimators": 200,
})
model.fit(dataset)

# 4. 生成预测信号
pred = model.predict(dataset, segment="test")

# 5. 回测配置
backtest_config = {
    "strategy": {
        "class":  "TopkDropoutStrategy",
        "kwargs": {
            "model":    model,
            "dataset":  dataset,
            "topk":     10,      # 多头持仓股票数
            "n_drop":   3,       # 每次换仓换掉最差的 3 只
        }
    },
    "executor": {
        "class":  "SimulatorExecutor",
        "kwargs": {
            "time_per_step":  "day",
            "generate_portfolio_metrics": True,
        }
    },
    "backtest": {
        "start_time": "2022-01-01",
        "end_time":   "2023-12-31",
        "account":    1_000_000,   # 初始资金 100万
        "benchmark":  "SPY",
        "exchange_kwargs": {
            "freq":       "day",
            "limit_threshold": None,
            "deal_price": "close",
            "open_cost":  0.0005,   # 0.5 bps 开仓成本
            "close_cost": 0.0015,   # 1.5 bps 平仓成本
            "min_cost":   5,
        }
    }
}

# 6. 运行回测
portfolio_metric, indicator = backtest(
    executor=backtest_config["executor"],
    strategy=backtest_config["strategy"],
    **backtest_config["backtest"]
)

# 7. 绩效分析
analysis_df = risk_analysis(portfolio_metric["1day"]["return"])
print(analysis_df)

# 8. 可视化报告（自动生成图表）
analysis_position.report_graph(portfolio_metric, show_notebook=True)
'''

print("qlib 完整回测代码模板：")
print(qlib_backtest_template)

In [ ]:
# 用我们自己的数据运行简化版回测，展示 qlib 的核心思路
# 使用第 5 步训练好的模型

def qlib_style_backtest(predictions_df, prices_df, 
                         topk=3, n_drop=1, cost_bps=10):
    """
    模仿 qlib TopkDropoutStrategy 的简化版回测。
    
    TopkDropoutStrategy：
    - 持有信号最高的 topk 只股票
    - 每次换仓时，只替换掉排名下降的 n_drop 只股票
    - 这样减少换手率（vs 每次重新选 topk 只）
    """
    all_dates = sorted(predictions_df['date'].unique())
    log_ret   = np.log(prices_df / prices_df.shift(1))
    
    holdings   = []   # 当前持有的股票
    port_rets  = []
    turnovers  = []
    
    for date in sorted(log_ret.index):
        if date < pd.Timestamp('2022-01-01'):
            continue
        
        # 当日预测
        day_pred = predictions_df[predictions_df['date'] == date]
        
        if len(day_pred) >= topk:
            ranked = day_pred.sort_values('pred', ascending=False)['ticker'].tolist()
            
            if not holdings:
                new_holdings = ranked[:topk]
                turnover = 1.0
            else:
                # TopkDropout：只换掉排名最差的 n_drop 只
                to_drop = [t for t in holdings if t not in ranked[:topk + n_drop]]
                to_drop = to_drop[:n_drop]
                
                to_add  = [t for t in ranked if t not in holdings]
                to_add  = to_add[:n_drop]
                
                new_holdings = [t for t in holdings if t not in to_drop] + to_add
                new_holdings = new_holdings[:topk]
                
                changed  = sum(1 for t in new_holdings if t not in holdings)
                turnover = changed / topk if topk > 0 else 0
            
            holdings = new_holdings
        
        cost = turnover * cost_bps * 1e-4 if holdings else 0
        
        if holdings and date in log_ret.index:
            day_log_ret = log_ret.loc[date]
            ret = np.mean([day_log_ret.get(t, 0) for t in holdings]) - cost
        else:
            ret = 0
        
        port_rets.append({'date': date, 'ret': ret, 'turnover': turnover,
                          'holdings': holdings.copy()})
    
    return pd.DataFrame(port_rets).set_index('date')

print("运行 TopkDropout 回测...")
port_qlib = qlib_style_backtest(test_df[['date','ticker','pred']], prices[TICKERS],
                                  topk=3, n_drop=1, cost_bps=10)

print(f"回测完成: {len(port_qlib)} 个交易日")

# 绩效报告
cum_ret = (1 + port_qlib['ret']).cumprod()
total   = cum_ret.iloc[-1] - 1
ann_ret = (1 + total) ** (252 / len(port_qlib)) - 1
ann_vol = port_qlib['ret'].std() * np.sqrt(252)
sharpe  = ann_ret / ann_vol
peak    = cum_ret.cummax()
max_dd  = ((cum_ret - peak) / peak).min()
turnover_avg = port_qlib['turnover'].mean()

print(f"\n回测结果（TopkDropout, topk=3, cost=10bps）：")
print(f"  年化收益:  {ann_ret*100:.2f}%")
print(f"  年化波动:  {ann_vol*100:.2f}%")
print(f"  Sharpe:    {sharpe:.3f}")
print(f"  最大回撤:  {max_dd*100:.2f}%")
print(f"  平均换手:  {turnover_avg*100:.1f}% 每换仓日")

## Step 7：实验管理 —— qlib Recorder

量化研究最常见的问题：**实验太多，记不清哪个参数对应哪个结果。**

qlib 内置了实验管理系统（基于 MLflow）：

```python
from qlib.workflow import R

with R.start(experiment_name='my_alpha'):
    R.log_params(num_leaves=64, lr=0.02)   # 记录超参数
    R.log_metric('IC', ic_mean)            # 记录指标
    R.save_objects(model=model)            # 保存模型
```

所有实验结果都存在本地数据库，可以：
- 按时间查看所有实验
- 比较不同超参数的效果
- 一键复现某次实验

In [ ]:
# 使用 qlib Recorder 记录实验
try:
    from qlib.workflow import R
    from qlib.workflow.record_temp import SignalRecord
    
    exp_name = 'vol5_lgbm_us'
    
    with R.start(experiment_name=exp_name):
        # 记录超参数
        R.log_params(**lgb_params)
        R.log_params(
            train_start='2019-01-01',
            train_end=str(TRAIN_END.date()),
            test_start=str(TRAIN_END.date()),
            test_end=str(TEST_END.date()),
            n_features=len(feature_names),
            feature_type='custom_qlib',
        )
        
        # 记录指标
        R.log_metrics(
            ic_mean=float(daily_ic.mean()),
            icir=float(daily_ic.mean() / daily_ic.std()),
            ic_pos_ratio=float((daily_ic > 0).mean()),
            backtest_sharpe=float(sharpe),
            backtest_ann_ret=float(ann_ret),
            backtest_maxdd=float(max_dd),
        )
        
        # 保存模型和预测
        import pickle
        R.save_objects(
            model=model,
            predictions=test_df[['date','ticker','pred','label_10d']].to_dict('records')
        )
        
        rid = R.get_recorder().id
        print(f"\n✅ 实验已记录！")
        print(f"   实验名称: {exp_name}")
        print(f"   记录 ID:  {rid}")
        print(f"   指标:     IC={daily_ic.mean():.4f}, ICIR={daily_ic.mean()/daily_ic.std():.3f}")
except Exception as e:
    print(f"Recorder 演示（需要完整 qlib 环境）: {type(e).__name__}")
    print("在完整环境中，所有实验参数和结果都会自动存储到本地数据库")
    print("可以用 qlib.workflow.R.list_experiments() 查看所有历史实验")

## Step 8：综合对比 —— 手写 vs qlib

经过 5 节课，我们学会了从头手写每个模块，也学会了用 qlib 系统化实现。来做个全面对比：

In [ ]:
comparison_table = """
╔══════════════════════════════════════════════════════════════════╗
║         手写 pandas/numpy    vs    qlib 框架                     ║
╠══════════╦═══════════════════════════╦═══════════════════════════╣
║  模块    ║   手写（Vol.1-4）          ║   qlib（Vol.5）            ║
╠══════════╬═══════════════════════════╬═══════════════════════════╣
║  数据    ║ yfinance + pandas         ║ D.features(expr)          ║
║          ║ 每次重复下载和处理          ║ 自动缓存，10x 速度          ║
╠══════════╬═══════════════════════════╬═══════════════════════════╣
║  因子    ║ 手写 Python 函数           ║ 表达式引擎 DSL              ║
║          ║ 逻辑分散，容易出 bug        ║ 统一语法，可复用             ║
╠══════════╬═══════════════════════════╬═══════════════════════════╣
║  模型    ║ 手写 Walk-Forward 训练     ║ model.fit(dataset)        ║
║          ║ 代码冗长                   ║ 统一接口，可换 LSTM/GRU     ║
╠══════════╬═══════════════════════════╬═══════════════════════════╣
║  回测    ║ 手写 portfolio_backtest()  ║ backtest(strategy, exec)  ║
║          ║ 难以考虑所有细节             ║ 涨跌停、T+1 等自动处理       ║
╠══════════╬═══════════════════════════╬═══════════════════════════╣
║  实验    ║ 结果手动记录到表格           ║ R.start() 自动存储          ║
║          ║ 复现困难                   ║ 一键复现任何历史实验          ║
╠══════════╬═══════════════════════════╬═══════════════════════════╣
║ 适用场景 ║ 学习原理、小规模实验          ║ 规模化研究、团队协作          ║
╚══════════╩═══════════════════════════╩═══════════════════════════╝

结论：先手写理解原理（Vol.1-4），再用 qlib 提升效率（Vol.5）
      在量化工作中，两者都需要，相辅相成。
"""
print(comparison_table)

In [ ]:
# 最终绩效可视化：qlib TopkDropout 策略 vs SPY
spy_ret = np.log(prices['^GSPC'] / prices['^GSPC'].shift(1)) if '^GSPC' in prices.columns else \
          np.log(prices['SPY'] / prices['SPY'].shift(1)) if 'SPY' in prices.columns else None

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# 净值曲线
ax = axes[0][0]
cum_port = (1 + port_qlib['ret']).cumprod()
ax.plot(cum_port.index, cum_port, 'steelblue', lw=2.5, label='qlib TopkDropout (topk=3)')
ax.axhline(1, color='gray', lw=0.8, ls=':')
ax.fill_between(cum_port.index, 1, cum_port,
                where=cum_port >= 1, alpha=0.12, color='steelblue')
ax.fill_between(cum_port.index, 1, cum_port,
                where=cum_port < 1, alpha=0.15, color='red')
ax.set_title('策略净值曲线')
ax.set_ylabel('Cumulative Return')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

# 回撤
ax2 = axes[0][1]
peak = cum_port.cummax()
dd   = (cum_port - peak) / peak
ax2.fill_between(dd.index, dd, 0, alpha=0.6, color='red')
ax2.set_title(f'回撤（最大 {dd.min()*100:.1f}%）')
ax2.set_ylabel('Drawdown')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

# IC 时序
ax3 = axes[1][0]
ax3.bar(daily_ic.index, daily_ic, alpha=0.4, color='steelblue', width=1)
ax3.plot(daily_ic.rolling(20).mean(), color='steelblue', lw=2)
ax3.axhline(0, color='black', lw=1)
ax3.axhline(daily_ic.mean(), color='red', ls='--', lw=1.5,
            label=f'均值={daily_ic.mean():.4f}')
ax3.set_title('日 Rank IC')
ax3.legend()
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

# 特征重要性
ax4 = axes[1][1]
importance = pd.Series(model.feature_importances_, index=feature_names)
importance = importance.sort_values(ascending=True)
colors = ['#d73027' if v < importance.median() else '#4575b4' for v in importance.values]
ax4.barh(importance.index, importance.values, color=colors, alpha=0.85)
ax4.set_title('特征重要性（qlib 表达式因子）')
ax4.set_xlabel('Importance')

plt.suptitle('qlib 完整工作流 —— 最终结果总览', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('qlib_final_results.png', bbox_inches='tight')
plt.show()

## 完整课程总结

```
量化交易从入门到 qlib  ——  完整知识体系

Vol.1  金融数据基础
    ├── OHLCV、收益率、统计特性（胖尾）
    ├── Sharpe Ratio、相关性
    └── 滚动统计 → 因子的原材料

Vol.2  Alpha 因子工程
    ├── 因子 = 对未来收益的截面预测
    ├── 5 大经典因子族（动量/反转/波动/成交量/技术）
    ├── IC 分析、ICIR、因子衰减
    └── 多因子合成（等权）

Vol.3  机器学习选股
    ├── Panel 数据集构建（截面排名标签）
    ├── Walk-Forward 时序验证（不可随机划分）
    ├── LightGBM：自动学习因子组合权重
    └── 过拟合检测（训练 vs 测试 IC 衰减）

Vol.4  回测基础
    ├── 多空组合构建（TopK Long + BottomK Short）
    ├── 核心指标：Sharpe、MaxDD、Calmar
    ├── 交易成本敏感性分析
    └── 收益归因（Alpha vs Beta）

Vol.5  qlib 工作流
    ├── 数据层：D.features() + 表达式引擎
    ├── DataHandler：统一特征管道
    ├── Model：LightGBM/LSTM 统一接口
    ├── 回测：TopkDropoutStrategy
    └── 实验管理：Recorder 自动存储
```

### 进阶学习路径

| 方向 | 内容 | 参考资源 |
|------|------|----------|
| 深度学习 | LSTM、Transformer、TCN 选股 | qlib 内置模型 |
| 组合优化 | 均值方差优化、Black-Litterman | PortfolioOptimizer |
| 风险管理 | 因子中性化、行业中性 | Barra 风险模型 |
| 另类数据 | 卫星图像、NLP 新闻、社交媒体 | OpenBB, Quandl |
| 高频交易 | L2 数据、做市策略 | Tick 数据 |

---

## 写在最后

量化交易是 **数学、金融、编程** 三者的交叉领域。  
本课程帮你建立了核心框架，但真正的学习还需要：

1. **大量实践**：用真实数据跑策略，感受噪音
2. **阅读论文**：因子 zoo 有 400+ 因子，大部分已发表
3. **谦逊**：市场比任何模型都复杂

祝你量化研究顺利！

---
*Vol.5 完 | 课程：量化交易从入门到 qlib*  
*By: 量化初学者友好课程*